In [1]:
# pip install tensorflow pandas numpy matplotlib seaborn scikit-learn opencv-python jupyter

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow
import os
import cv2
import random

In [3]:
metadata = pd.read_csv(r"data\HAM10000_metadata.csv")

In [4]:
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [5]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  str    
 1   image_id      10015 non-null  str    
 2   dx            10015 non-null  str    
 3   dx_type       10015 non-null  str    
 4   age           9958 non-null   float64
 5   sex           10015 non-null  str    
 6   localization  10015 non-null  str    
dtypes: float64(1), str(6)
memory usage: 547.8 KB


In [6]:
print(metadata.isnull().sum())

lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64


Only age has missing values, so I will likely have to the median

Cases include a representative collection of all important diagnostic categories in the realm of pigmented lesions: Actinic keratoses and intraepithelial carcinoma / Bowen's disease (akiec), basal cell carcinoma (bcc), benign keratosis-like lesions (solar lentigines / seborrheic keratoses and lichen-planus like keratoses, bkl), dermatofibroma (df), melanoma (mel), melanocytic nevi (nv) and vascular lesions (angiomas, angiokeratomas, pyogenic granulomas and hemorrhage, vasc).

More than 50% of lesions are confirmed through histopathology (histo), the ground truth for the rest of the cases is either follow-up examination (follow_up), expert consensus (consensus), or confirmation by in-vivo confocal microscopy (confocal). The dataset includes lesions with multiple images, which can be tracked by the lesion_id-column within the HAM10000_metadata file.  

---
The target feature is dx (diagnosis)

In [7]:
class_counts = metadata['dx'].value_counts()
print(class_counts)

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


- akiec (Actinic keratoses and intraepithelial carcinoma / Bowen's disease): "pre-cancer" or early cancer cells
- bcc (basal cell carcinoma): cancer that typically occurs througout the whole body
- bkl (benign keratosis-like lesions): harmless
- df (dermatofibroma): not cancer
- mel (melanoma): cancer
- nv (melanocytic nevi): birthmark/mole has the potential to become cancerous
- vasc (vascular lesions): mostly benign but has the potenial to be cancerous
    
I see that there are roughly three categories: cancerous, completely benign, and potential to become cancerous. It am considering combining them into those groups because of how unbalanced the classes are

In [8]:
# plt.figure(figsize=(10,5))
# sns.barplot(x=class_counts.index, y=class_counts.values)
# plt.title('Number of images per diagnosis')
# plt.xlabel('Diagnosis')
# plt.ylabel('Count')
# plt.show()

In [9]:
image_dir = r'data\images'
metadata['path'] = metadata['image_id'].apply(lambda x: os.path.join(image_dir, f"{x}.jpg"))

In [10]:
# Check that files exist
sample_paths = metadata['path'].head()
for p in sample_paths:
    print(f"Exists {p}? {os.path.exists(p)}")

Exists data\images\ISIC_0027419.jpg? True
Exists data\images\ISIC_0025030.jpg? True
Exists data\images\ISIC_0026769.jpg? True
Exists data\images\ISIC_0025661.jpg? True
Exists data\images\ISIC_0031633.jpg? True


In [16]:
# Display a few random images from each class
def display_rand_image_from_each_class():
    fig, axes = plt.subplots(2, 4, figsize=(12,6))
    axes = axes.flatten()
    
    for i, dx_class in enumerate(class_counts.index[:7]):
        # Get all rows of this class
        class_subset = metadata[metadata['dx'] == dx_class]
        # Pick a random image
        row = class_subset.sample(1).iloc[0]
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # OpenCV loads as BGR, convert to RGB
        axes[i].imshow(img)
        axes[i].set_title(dx_class)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# display_rand_image_from_each_class()